# Clase 44 — Cómo funciona ChatGPT · Solucionario

**Solo para el profesor.** No se sube a Classroom ni se proyecta antes de dictar la clase.

Cada bloque usa las funciones de la primera celda (sin librerías externas). Las salidas con sorteo se muestran con una semilla fija **solo como ejemplo**: en clase cada ejecución cambia.

## ⚙️ Celda preparada

La misma celda inicial que ven los estudiantes.

In [ ]:
#@title 🔧 Ejecuta esta celda primero (deja listas las funciones de hoy; no hace falta editarla)
import random
from collections import Counter

# Nuestra «base de datos» de juguete: el chat del curso
texto_base = "nos vemos en el patio . nos vemos en el patio . nos vemos en la plaza . ya llegué al casino ."


def _contar_siguientes(texto):
    palabras = texto.split()
    siguientes = {}
    for palabra, siguiente in zip(palabras, palabras[1:]):
        siguientes.setdefault(palabra, Counter())[siguiente] += 1
    return siguientes


def _conocidas(texto):
    return sorted(set(texto.split()))


def _sacar_ficha(tabla, conocidas, palabra):
    # Con bolsa propia: se saca una ficha al azar (más fichas = más probable).
    # Palabra que nunca vio: se saca al azar entre las palabras que sí conoce.
    if palabra in tabla:
        opciones = tabla[palabra]
        return random.choices(list(opciones), weights=list(opciones.values()))[0]
    return random.choice(conocidas)


def ver_tokens(texto):
    tokens = texto.split()
    print("Texto:", texto)
    print("Tokens (en nuestro mini modelo, cada palabra es un token):")
    print(" | ".join(tokens))
    print("Cantidad de tokens:", len(tokens))


def ver_tokens_modelos_reales():
    print("Ilustración: así podría cortar un modelo grande las palabras largas o poco comunes")
    ejemplos = [("casa", ["casa"]), ("inteligencia", ["inteli", "gencia"]), ("murciélago", ["mur", "ciél", "ago"])]
    for palabra, trozos in ejemplos:
        plural = "s" if len(trozos) > 1 else ""
        print(f"  {palabra:14s} → {' | '.join(trozos)}   ({len(trozos)} token{plural})")
    print("(Cada modelo corta a su manera: esto es solo una ilustración.)")


def mostrar_probabilidades(texto, palabra):
    tabla = _contar_siguientes(texto)
    if palabra not in tabla:
        print(f"«{palabra}» nunca apareció en el texto: no tiene bolsa propia.")
        print(f"Si le preguntamos qué viene después, el modelo saca una ficha AL AZAR entre las {len(_conocidas(texto))} palabras que sí conoce.")
        return
    total = sum(tabla[palabra].values())
    print(f"Bolsa de «{palabra}»: {total} fichas")
    for siguiente, veces in tabla[palabra].most_common():
        print(f"  «{siguiente}»: {veces} de {total} = {veces / total:.0%}")


def sacar_fichas(texto, palabra, cantidad=1):
    tabla = _contar_siguientes(texto)
    conocidas = _conocidas(texto)
    sacadas = [_sacar_ficha(tabla, conocidas, palabra) for _ in range(cantidad)]
    if palabra in tabla:
        print(f"Saqué {cantidad} ficha(s) de la bolsa de «{palabra}»:")
    else:
        print(f"«{palabra}» nunca apareció: saqué {cantidad} ficha(s) AL AZAR entre las {len(conocidas)} palabras que sí conoce:")
    print("  " + " ".join(sacadas))
    if cantidad > 1:
        conteo = Counter(sacadas)
        print("  Conteo: " + ", ".join(f"«{p}» {n}" for p, n in conteo.most_common()))


def _frase(texto, inicio, largo):
    tabla = _contar_siguientes(texto)
    conocidas = _conocidas(texto)
    frase = [inicio]
    while len(frase) < largo:
        frase.append(_sacar_ficha(tabla, conocidas, frase[-1]))
    return " ".join(frase)


def generar_frase(texto, inicio, largo=10):
    print(_frase(texto, inicio, largo))


def generar_varias(texto, inicio, largo=10, cantidad=5):
    for numero in range(1, cantidad + 1):
        print(f"{numero}: {_frase(texto, inicio, largo)}")


print("Funciones listas ✅")

## 🧠 Haz Ahora

Chat: «nos vemos en el recreo / nos vemos en el kiosco / llego en la micro».

1. Sugeriría **`vemos`**: las dos veces que aparece «nos» la palabra siguiente es «vemos».
2. «en» aparece **3 veces**.
3. De esas 3 veces, **2** vienen seguidas de «el» (y 1 de «la»).
4. La calculadora **sí** da siempre 56 (determinística); el amigo **no** recomienda siempre lo mismo (varía). Respuesta válida: «la calculadora es siempre igual; con la persona/amigo la respuesta puede cambiar». Aún no se nombran los términos: **estocástico** ya está en la definición de LLM (concepto 1) y se desarrolla en el concepto 3.

**Línea de toda la clase (Diego):** el modelo chiquito y el gigante son **estocásticos** (comportamiento que no se puede predecir con total exactitud: hay azar). Lo que responde un modelo es **azaroso** y depende de las **probabilidades con que se entrenó**. Imagen única: la **bolsa de fichas** (se saca una ficha al azar; más fichas = más probable, sin garantía). «Determinístico» solo aparece como contraste con la calculadora.

## 📘 ICN — código completo y salidas

### Concepto 1 · Nuestra «base de datos»

**Salida:**

```text
Nuestra base de datos:
nos vemos en el patio . nos vemos en el patio . nos vemos en la plaza . ya llegué al casino .
Palabras en el texto: 23
```

Lo esencial: **LLM = Large Language Model** (modelo de lenguaje grande): aprendió de textos a predecir la siguiente palabra y es **estocástico** (tiene azar). El teclado es uno chiquito y ChatGPT uno gigante; los dos tienen azar.

In [ ]:
print("Nuestra base de datos:")
print(texto_base)
print("Palabras en el texto:", len(texto_base.split()))

### Concepto 2 · Tokens (mini modelo)

**Salida:**

```text
Texto: nos vemos en el patio . nos vemos en el patio . nos vemos en la plaza . ya llegué al casino .
Tokens (en nuestro mini modelo, cada palabra es un token):
nos | vemos | en | el | patio | . | nos | vemos | en | el | patio | . | nos | vemos | en | la | plaza | . | ya | llegué | al | casino | .
Cantidad de tokens: 23
```

In [ ]:
ver_tokens(texto_base)

### Concepto 2 · Tokens (modelos reales, ilustración)

**Salida:**

```text
Ilustración: así podría cortar un modelo grande las palabras largas o poco comunes
  casa           → casa   (1 token)
  inteligencia   → inteli | gencia   (2 tokens)
  murciélago     → mur | ciél | ago   (3 tokens)
(Cada modelo corta a su manera: esto es solo una ilustración.)
```

Es una **ilustración**: cada modelo corta a su manera y no se afirma cómo lo hace ningún producto concreto. En el mini modelo, 1 palabra = 1 token (y el `.` también).

In [ ]:
ver_tokens_modelos_reales()

### Concepto 3 · La bolsa de `en`

**Salida:**

```text
Bolsa de «en»: 3 fichas
  «el»: 2 de 3 = 67%
  «la»: 1 de 3 = 33%
```

Bolsa de `en`: 2 fichas `el` y 1 ficha `la` → `el` 2 de 3 (≈ 67 %), `la` 1 de 3 (≈ 33 %). Probabilidad = proporción de fichas.

In [ ]:
mostrar_probabilidades(texto_base, "en")

### Concepto 3 · Sacar 10 fichas (azar)

**Salida:**

```text
Saqué 10 ficha(s) de la bolsa de «en»:
  el el el el el el la la la el
  Conteo: «el» 7, «la» 3
```

**Ejemplo con semilla fija; en clase cada ejecución varía.** `el` sale más seguido, pero `la` también puede salir, y el conteo cambia de una ejecución a otra (puede ser 7 y 3, 5 y 5, 9 y 1…): es **estocástico**. Frase a evitar: «siempre elige la más probable».

In [ ]:
sacar_fichas(texto_base, "en", 10)

### Concepto 3 · Palabra que nunca vio

**Salida:**

```text
«biblioteca» nunca apareció en el texto: no tiene bolsa propia.
Si le preguntamos qué viene después, el modelo saca una ficha AL AZAR entre las 12 palabras que sí conoce.
«biblioteca» nunca apareció: saqué 5 ficha(s) AL AZAR entre las 12 palabras que sí conoce:
  . al al la casino
  Conteo: «al» 2, «.» 1, «la» 1, «casino» 1
```

No tiene bolsa propia, pero **responde igual**: saca una ficha **al azar entre las 12 palabras que sí conoce**, sin ninguna base. Lo que responde un modelo es azaroso y depende de con qué se entrenó (ejemplo con semilla fija).

In [ ]:
mostrar_probabilidades(texto_base, "biblioteca")
sacar_fichas(texto_base, "biblioteca", 5)

### Concepto 3 · Generar 5 frases

**Salida:**

```text
1: nos vemos en el patio . nos vemos en el
2: nos vemos en el patio . nos vemos en la
3: nos vemos en la plaza . nos vemos en la
4: nos vemos en la plaza . nos vemos en la
5: nos vemos en el patio . nos vemos en el
```

**Ejemplo con semilla fija; en clase cada ejecución varía.** Los caminos se separan tras `en` (`el` 2/3 o `la` 1/3) y tras `.` (`nos` 2/3 o `ya` 1/3). Es la misma bolsa de fichas, usada una y otra vez.

In [ ]:
generar_varias(texto_base, "nos", 10, 5)

### 🌐 Concepto 4 · Evidenciarlo en la práctica (prompt real en vivo) — qué esperar

Prompt (2 veces, conversaciones nuevas): «Explica las diferencias entre ChatGPT, Claude y Gemini». **Las respuestas las genera ChatGPT y varían**, así que no hay respuesta única. Rúbrica orientadora para las celdas de los estudiantes:

| Componente | ✅ completo | ⚠️ parcial | ❌ cero |
|---|---|---|---|
| Qué cambió / qué se mantuvo | nombra al menos un cambio y una constante, **con cita de ambas respuestas** | nombra sin cita, o cita solo una | sin evidencia |
| Por qué | dice que el modelo es **estocástico** (su comportamiento no se puede predecir con total exactitud, hay azar): en cada paso saca una ficha al azar según probabilidades y no siempre la misma, y el camino se acumula | «es aleatorio» sin ligarlo a la siguiente palabra | explicación incorrecta |

**Aprovecha para decir:** las respuestas sobre cada producto pueden estar desactualizadas o ser erróneas; «revisar si algo es confiable» (sin nombrar las 4D). Típicamente se mantiene la estructura/idea general y cambian la redacción, el orden y los ejemplos. *(Referencia B2 de la Prueba 2.)*

## ✅ Verificamos lo aprendido — respuestas esperadas (el colega pregunta a dos personas por concepto; 4 conceptos)

| Concepto | Pregunta | Respuesta esperada |
|---|---|---|
| 1 · LLM | ¿Qué significa LLM y por qué es estocástico? | *Large Language Model* (modelo de lenguaje grande): aprendió de textos a predecir la siguiente palabra; es estocástico porque su comportamiento no se puede predecir con total exactitud (tiene azar). |
| 1 · LLM | ¿En qué se parece el teclado a ChatGPT? | Los dos predicen la siguiente palabra y tienen azar; el teclado es uno chiquito y ChatGPT uno gigante. |
| 2 · Tokens | ¿Qué es un token? ¿Cuántos tiene la base de datos? | Un trozo de texto (una ficha) que el modelo lee como una sola pieza; en el mini modelo, **23** (cada palabra y cada `.`). |
| 2 · Tokens | En los modelos grandes, ¿un token es siempre una palabra? | No: las palabras largas o poco comunes se cortan en varios trozos. |
| 3 · Bolsa y azar | En la bolsa de `en` (2 `el`, 1 `la`): ¿probabilidad de cada una? ¿Por qué puede salir `la`? | `el` 2 de 3 ≈ 67 %; `la` 1 de 3 ≈ 33 %. Puede salir `la` porque el modelo saca una ficha **al azar**: la más probable sale más seguido, pero no está garantizada. |
| 3 · Bolsa y azar | Palabra nunca vista: ¿qué hace? ¿De qué depende lo que responde? | Saca una ficha al azar entre las palabras que sí conoce. Lo que responde depende del azar y de las probabilidades con que se entrenó. |
| 4 · Evidencia | ¿Qué cambió y qué se mantuvo entre las dos respuestas? ¿Cómo lo explica que sea estocástico? | Cambian redacción, orden y ejemplos; se mantiene la idea general/estructura. Se explica porque en cada paso se saca una ficha al azar según probabilidades y un camino distinto se va acumulando. |
| 4 · Evidencia | ¿Por qué dos respuestas de ChatGPT a la misma pregunta pueden ser distintas? | Porque es estocástico: lo que responde es azaroso y depende de las probabilidades con que se entrenó. |

**Conteo de 2 minutos en la pizarra (concepto 3):** escribir el texto del patio, subrayar cada `en`, anotar lo que viene después (`el`, `el`, `la`), contar y dividir: 2 de 3 ≈ 67 %. La práctica guiada (réplica de la Parte D de la Prueba 1) se responde **a mano** antes de ejecutar las celdas de comprobación.

## 🧩 Práctica guiada (réplica de la Parte D) — solución

### Texto del horario

**Salida:**

```text
hoy toca mate . hoy toca mate . hoy hay prueba . hoy toca inglés .
```

In [ ]:
texto_tarea = "hoy toca mate . hoy toca mate . hoy hay prueba . hoy toca inglés ."
print(texto_tarea)

### D1 · tras `toca`

**Salida:**

```text
Bolsa de «toca»: 3 fichas
  «mate»: 2 de 3 = 67%
  «inglés»: 1 de 3 = 33%
```

**`mate`, 2 de 3 veces** (`inglés`, 1 de 3). ✅ palabra + cuántas veces · ⚠️ solo la palabra · ❌ sin respuesta.

In [ ]:
texto_tarea = "hoy toca mate . hoy toca mate . hoy hay prueba . hoy toca inglés ."
mostrar_probabilidades(texto_tarea, "toca")

### D2 · tras `hoy`

**Salida:**

```text
Bolsa de «hoy»: 4 fichas
  «toca»: 3 de 4 = 75%
  «hay»: 1 de 4 = 25%
```

**`toca`, 3 de 4 = 75 %** (`hay`, 1 de 4). El total es 4 (las veces que apareció `hoy`), no las 12 palabras del texto. ✅ palabra + proporción · ⚠️ solo la palabra · ❌ sin respuesta.

In [ ]:
texto_tarea = "hoy toca mate . hoy toca mate . hoy hay prueba . hoy toca inglés ."
mostrar_probabilidades(texto_tarea, "hoy")

### D3 y D4 · `feriado`

**Salida:**

```text
«feriado» nunca apareció en el texto: no tiene bolsa propia.
Si le preguntamos qué viene después, el modelo saca una ficha AL AZAR entre las 7 palabras que sí conoce.
«feriado» nunca apareció: saqué 3 ficha(s) AL AZAR entre las 7 palabras que sí conoce:
  mate hoy prueba
  Conteo: «mate» 1, «hoy» 1, «prueba» 1
```

**D3:** **no** puede usar su regla: no hay nada que contar, `feriado` no tiene bolsa propia. ✅ «no» + razón (nunca la vio) · ⚠️ «no» sin razón · ❌ «sí» sin más. *(Se acepta «sí, pero al azar» si aclara que no tiene base: es consistente con D4.)*

**D4:** igual tendría que decir algo: **saca una ficha al azar entre las palabras que sí conoce**; la respuesta es **azarosa** (no predecible), depende de con qué se entrenó (ejemplo con semilla fija).

**Revisión en vivo:** una pareja explica al frente cómo contó a mano. Es una réplica de la Parte D de la Prueba 1 (mismo formato, otro texto).

In [ ]:
texto_tarea = "hoy toca mate . hoy toca mate . hoy hay prueba . hoy toca inglés ."
mostrar_probabilidades(texto_tarea, "feriado")
sacar_fichas(texto_tarea, "feriado", 3)

## 🚀 Ejercicios extra — soluciones

### Ejercicio 0a · Cambiar una palabra

**Salida:**

```text
Bolsa de «en»: 3 fichas
  «la»: 2 de 3 = 67%
  «el»: 1 de 3 = 33%
```

Con «en la plaza» en la primera oración, la bolsa de `en` queda con 1 ficha `el` y 2 fichas `la`: ahora `la` es la más probable (2 de 3). Cualquier cambio equivalente vale; lo importante es que anticipen que un cambio pequeño en la «base de datos» cambia las probabilidades.

In [ ]:
texto_modificado = "nos vemos en la plaza . nos vemos en el patio . nos vemos en la plaza . ya llegué al casino ."
mostrar_probabilidades(texto_modificado, "en")

### Ejercicio 1 · Cafetería

**Salida:**

```text
Bolsa de «pido»: 4 fichas
  «pan»: 3 de 4 = 75%
  «jugo»: 1 de 4 = 25%
Bolsa de «con»: 3 fichas
  «palta»: 2 de 3 = 67%
  «queso»: 1 de 3 = 33%
«empanada» nunca apareció en el texto: no tiene bolsa propia.
Si le preguntamos qué viene después, el modelo saca una ficha AL AZAR entre las 7 palabras que sí conoce.
```

(a) `pan`, 3 de 4. (b) tras `con`: `palta`, 2 de 3 ≈ 67 %. (c) `empanada` nunca apareció: no tiene bolsa propia, el modelo saca una ficha al azar entre las palabras que sí conoce.

In [ ]:
texto_cafeteria = "pido pan con palta . pido pan con palta . pido pan con queso . pido jugo ."
mostrar_probabilidades(texto_cafeteria, "pido")
mostrar_probabilidades(texto_cafeteria, "con")
mostrar_probabilidades(texto_cafeteria, "empanada")

### Ejercicio 2 · Frases distintas (desafío)

**Salida:**

```text
1: nos vemos en la plaza . nos vemos en el
2: nos vemos en el patio . ya llegué al casino
3: nos vemos en el patio . nos vemos en el
4: nos vemos en la plaza . nos vemos en el
5: nos vemos en la plaza . nos vemos en la
```

**Ejemplo con semilla fija.** Casi siempre salen 4-5 frases distintas; que las 5 coincidan es posible pero muy poco probable. Conclusión esperada: en un sistema estocástico cada sorteo es nuevo (hay azar), por eso los resultados cambian de una ejecución a otra.

In [ ]:
generar_varias(texto_base, "nos", 10, 5)

## 🎫 Ticket de salida — preguntas y respuestas

Respuestas correctas: **C, D, B, A**. Las preguntas **no están en el Colab de estudiantes**: se proyectan en `Ticket de Salida.pptx` y se responden en el Google Form (`https://forms.gle/sjRpbgmQzrpkEBsH9`). Respuestas también en `Ticket de Salida Respuestas.json`.

### Pregunta 1 — Qué es un modelo de lenguaje (concepto 1)

```text
Teclado del celular:  escribes «nos vemos en el»  →  sugiere «patio»
ChatGPT:  escribes «Cuéntame un cuento sobre»  →  continúa palabra por palabra
```

Los dos de arriba son modelos de lenguaje (LLM). ¿Qué es un LLM?

- **A)** Una base de datos con todas las respuestas escritas de antemano.
- **B)** Un buscador que encuentra páginas de internet con la respuesta.
- **C)** Un modelo que aprendió de textos a predecir la siguiente palabra y que tiene azar (es estocástico).
- **D)** Un programa que entiende lo que sentimos y por eso responde.

**Respuesta correcta: C.** LLM significa Large Language Model: aprendió, leyendo textos, a predecir la siguiente palabra, y es estocástico (su comportamiento no se puede predecir con total exactitud: tiene azar). El teclado del celular es uno chiquito y ChatGPT uno gigante.

### Pregunta 2 — Qué es un token (concepto 2)

```text
inteligencia  →  inteli | gencia
murciélago    →  mur | ciél | ago
```

Un modelo de lenguaje parte las palabras en trozos como los de arriba. ¿Qué es un token?

- **A)** Una palabra que el modelo nunca vio.
- **B)** Un número que mide qué tan seguro está el modelo.
- **C)** El texto completo con el que se entrenó el modelo.
- **D)** Un trozo de texto, a veces una palabra completa y a veces parte de ella, que el modelo lee como una sola pieza.

**Respuesta correcta: D.** Un token es un trozo de texto (a veces una palabra entera, a veces parte de una) que el modelo trata como una sola pieza.

### Pregunta 3 — Predecir y proporción (concepto 3)

```text
voy al kiosco . voy al kiosco . voy al baño . voy a clases .
```

Un mini modelo de lenguaje aprendió de este texto (cada `.` termina una oración). Su regla: después de una palabra, predice la que más veces la siguió. ¿Qué predice después de `voy` y con qué proporción?

- **A)** `kiosco`, 2 de 4 veces.
- **B)** `al`, 3 de 4 veces.
- **C)** `a`, 1 de 4 veces.
- **D)** `al`, 3 de 16 veces (las 16 palabras del texto).

**Respuesta correcta: B.** `voy` apareció 4 veces con algo después: `al` 3 veces y `a` 1. La proporción es 3 de 4 (el total es de las veces que apareció `voy`, no del largo del texto).

### Pregunta 4 — Por qué cambia la respuesta (concepto 4)

```text
Pregunta (enviada 2 veces, en conversaciones nuevas):
«Recomiéndame un panorama para el fin de semana»

Respuesta 1: «Te recomiendo ir al cine...»
Respuesta 2: «Te recomiendo caminar por el parque...»
```

Con exactamente la misma pregunta, ChatGPT dio respuestas distintas. ¿Por qué?

- **A)** Porque un modelo de lenguaje es estocástico: lo que responde depende del azar y de las probabilidades con que se entrenó.
- **B)** Porque ChatGPT se equivocó y una de las dos respuestas es incorrecta.
- **C)** Porque ChatGPT cambia de modelo cada vez que se le escribe.
- **D)** Porque no entendió la pregunta la primera vez.

**Respuesta correcta: A.** Un LLM es estocástico: lo que responde es azaroso y depende de las probabilidades con que se entrenó. En cada paso saca una ficha al azar de su bolsa: la más probable sale más seguido, pero no siempre; con la misma entrada el camino puede cambiar.